In [ ]:
# ==============================================================================
# STEP 3: FINAL MERGE & APP PREPARATION
# 
# DESCRIPTION:
# This script takes the untouched metadata from the Phase 1 master dataset and 
# merges it with the finalized 'category_human' column from your Excel audit.
# It outputs the final, app-ready CSV into the data/processed/ directory.
# ==============================================================================

import os
import sys
import pandas as pd

print("\n" + "="*60)
print(" STEP 3: BUILDING FINAL APPLICATION DATASET ")
print("="*60)

# --- 1. SETUP & DIRECTORIES ---
notebook_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
interim_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "interim"))
processed_dir = os.path.abspath(os.path.join(notebook_dir, "..", "..", "data", "processed"))

os.makedirs(processed_dir, exist_ok=True)

# File Paths
master_file = os.path.join(interim_dir, "master_ontology_dataset.csv")

# NOTE: Update this filename if you saved your Excel file with a different date/name
human_review_file = os.path.join(interim_dir, "human_category_review.xlsx") 
final_app_file = os.path.join(processed_dir, "ontology_app_dataset.csv")

# --- 2. LOAD DATASETS ---
try:
    print("[*] Loading Master Dataset...")
    master_df = pd.read_csv(master_file, dtype={'CURIE': str})
    
    print(f"[*] Loading Human Reviewed Data from {os.path.basename(human_review_file)}...")
    review_df = pd.read_excel(human_review_file, dtype={'CURIE': str})
except FileNotFoundError as e:
    print(f"[!] CRITICAL ERROR: Could not find required file. {e}")
    sys.exit(1)

# --- 3. MERGE DATA ---
print("[*] Merging datasets...")
# We only extract the categorization columns from the Excel file to prevent 
# accidental overwriting of the master metadata (Labels, Synonyms, etc.)
cols_to_merge = ['CURIE', 'category', 'confidence', 'category_human']

# Check if required columns exist in the Excel file
missing_cols = [col for col in cols_to_merge if col not in review_df.columns]
if missing_cols:
    print(f"[!] ERROR: The Excel file is missing the following required columns: {missing_cols}")
    sys.exit(1)

# Left join ensures we keep all Master rows, even if they aren't in the Excel file yet
final_df = master_df.merge(review_df[cols_to_merge], on='CURIE', how='left')

# --- 4. DETERMINE REVIEW STATUS ---
def determine_status(row):
    """
    Evaluates the categorization state of each concept based on the human audit protocol.
    """
    human_val = str(row['category_human']).strip() if pd.notna(row['category_human']) else ""
    ai_val = str(row['category']).strip() if pd.notna(row['category']) else ""
    
    # 1. Human Audited: The category_human column has data
    if human_val and human_val.lower() != 'nan':
        return human_val, "Human Reviewed"
    
    # 2. Pending Review: AI has categorized it, but the human hasn't copied/audited it yet
    elif ai_val and ai_val.lower() != 'nan':
        return ai_val, "Pending Human Review"
    
    # 3. Pending API: Brand new row from Phase 1 that hasn't run through Phase 2 yet
    else:
        return "Uncategorized", "Pending API"

print("[*] Applying review status logic...")
# Apply the logic across the dataframe
final_df[['working_category', 'review_status']] = final_df.apply(
    lambda r: pd.Series(determine_status(r)), axis=1
)

# Optional: Drop the intermediate AI columns if you only want the final category in the app
final_df = final_df.drop(columns=['category', 'confidence', 'category_human'])

# --- 5. EXPORT & REPORT ---
print(f"[*] Exporting application dataset to {os.path.basename(final_app_file)}...")
final_df.to_csv(final_app_file, index=False, encoding='utf-8-sig')

print("\n" + "="*60)
print(" DATASET STATUS SUMMARY ")
print("="*60)
status_counts = final_df['review_status'].value_counts()
for status, count in status_counts.items():
    print(f" - {status:<25}: {count:,} concepts")

print(f"\n[COMPLETE] Final dataset ready for downstream visualization.")
print("="*60)